## Data Preparation

- Feature Engineering (abgeleitet aus den Befunden in `03_Visualisation`)
- Speichern als Parquet

Grundlage ist `credit_dtypes_pcd.parquet` aus `02_Datatypes.ipynb` (Datentypen bereits angepasst).


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as numpy
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_parquet('../data/processed/credit_dtypes_pcd.parquet')


---
## Datenbereinigung

Bevor die Daten gespeichert werden, entfernen wir Zeilen mit unmöglichen Werten.

Die Erkenntnisse dazu stammen aus:
- **01_Data_Business_Understanding**: Prüfung auf komplett doppelte Zeilen (`df.duplicated().sum()`)
- **03_Visualisation_Carl**, Abschnitt *„3. Outliers in the Numerical Features“*: Boxplots, IQR-Ausreißertabelle und die extremsten Fälle (`nlargest`) bei `person_age` und `person_emp_length`

Wir entfernen bewusst nur Werte, die **unmöglich** sind, nicht alle statistischen Ausreißer.



In [ ]:
anzahl_vorher = len(df)
print(f'Zeilen vor der Bereinigung: {anzahl_vorher}')

### 1. Unrealistisches Alter entfernen

In 03_Visualisation_Carl zeigen der Boxplot von `person_age` und die Tabelle `nlargest(5, "person_age")` Personen mit einem Alter von über 100 Jahren.

Das sind offensichtlich Eingabefehler. Wir entfernen alle Zeilen mit `person_age > 100`.

In [ ]:
maske_alter = df['person_age'] > 100

display(df.loc[maske_alter, ['person_age', 'person_income', 'person_emp_length']])

df = df[~maske_alter]
print(f'Entfernt wegen Alter über 100: {maske_alter.sum()}')

### 3. Unmögliche Beschäftigungsdauer entfernen

In 03_Visualisation zeigt `nlargest(5, "person_emp_length")` Personen, deren Beschäftigungsdauer fast so hoch oder höher ist als ihr Alter.

Regel: Wer mit Alter minus Beschäftigungsdauer unter 14 Jahren angefangen hätte zu arbeiten, hat einen unmöglichen Wert.

Fehlende Werte (NaN) bei `person_emp_length` bleiben drin. Sie werden später im Modelling mit dem Median aufgefüllt (siehe 04_Modelling_Basic).

In [ ]:
MIN_ALTER_ARBEITSBEGINN = 14

arbeitsbeginn = df['person_age'] - df['person_emp_length']
maske_job = arbeitsbeginn < MIN_ALTER_ARBEITSBEGINN   # NaN ergibt False, bleibt also drin

display(df.loc[maske_job, ['person_age', 'person_emp_length']])

df = df[~maske_job]
print(f'Entfernt wegen unmöglicher Beschäftigungsdauer: {maske_job.sum()}')

### 5. Zinssatz (`loan_int_rate`) entfernen

Der Zinssatz wird von der Bank erst **nach** der Risikobewertung festgelegt: Wer riskanter eingeschätzt wird, bekommt einen höheren Zins. Bei einem neuen Kreditantrag steht dieser Wert also noch gar nicht zur Verfügung.

Würden wir ihn im Modell behalten, nutzt das Modell indirekt das Ergebnis der bisherigen manuellen Risikoprüfung. Das ist eine Form von **Data Leakage**: Das Modell wirkt im Test besser, als es in der echten Anwendung wäre.

Hinweise darauf gab es schon vorher:
- In der Korrelationsanalyse in 03 ist der Zinssatz eines der Merkmale mit der stärksten Korrelation zum Zahlungsausfall, was bei einem nachträglich vergebenen Wert zu erwarten ist.
- Im Modell-Tuning haben wir das Modell mit und ohne Zinssatz verglichen.

Nebeneffekt: Damit entfallen auch die fehlenden Werte in dieser Spalte.

In [ ]:
df = df.drop(columns=['loan_int_rate'])

print('Verbleibende Spalten:')
print(list(df.columns))

---
### Feature Engineering

- **loan_to_income**: Kredithöhe im Verhältnis zum Jahreseinkommen. Das stärkste
  Einzelsignal: Ab einem Verhältnis von 0,3 fallen 69 % der Kredite aus, darunter 15 %.

- **person_emp_length_missing**: Markiert Anträge ohne Angabe zur Beschäftigungsdauer.
  Hier fallen 31,5 % aus statt 21,5 %. Durch die spätere Median-Imputation würde
  diese Information sonst verloren gehen.

In [ ]:
# Verhältnis Kredithöhe zu Einkommen (Einkommen ist nie 0, Minimum 4.000)
df['loan_to_income'] = df['loan_amnt'] / df['person_income']

# Fehlende Beschäftigungsdauer als eigenes Merkmal
df['person_emp_length_missing'] = df['person_emp_length'].isna().astype('boolean')

---
### Save the Data in Parquet File

In [ ]:
ordner = Path('../data/processed')
ordner.mkdir(parents=True, exist_ok=True)

df.to_parquet(ordner / 'credit_transformed.parquet', index=False)  # Originalnamen + Features
print('Gespeichert:', df.shape)